In [8]:
""" Create the observational data set after controlling for pretreatment effects
"""
import pandas as pd
import os
import numpy as np
import statsmodels.api as sm

In [9]:
obs_paul = pd.read_excel(os.path.join(os.environ['HOME'],
    'Git', 'phenology_elm', "SPRUCE C Budget Summary 28Apr2022EXP.xlsx"),
    sheet_name="DataForPythonRead", skiprows=1,  engine="openpyxl")
obs_paul = obs_paul.loc[obs_paul["Year"] != 2020, :]
obs_paul = obs_paul.set_index(['Plot', 'Year']).sort_index()

obs_verity = pd.read_csv(os.path.join(os.environ['HOME'],
    'Git', 'phenology_elm', 'SalmonSPRUCE_2016to2021_AbovegroundPFT_CNPbudget_20240208.csv'))
# match by plot and year to temperature
obs_verity['Plot'] = [f'P{p:02d}' for p in obs_verity['Plot']]
obs_verity = obs_verity.set_index(['Plot', 'Year', 'PFT']).unstack()
obs_verity = obs_verity.loc[:, (slice(None), 
                                ['Sphagnum', 'evergreen conifer', 'deciduous conifer',  
                                    'shrub'])]
obs_verity2 = obs_verity.loc[:, ['ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear',
                                 'Pretrt_ABGbiomass_gCperm2', 'Pretrt_ABGnpp_gCperm2peryear']]
obs_data = pd.concat([obs_paul, obs_verity2], axis = 1, join = 'outer')
obs_data = obs_data.reset_index()

# Merge 0 and Amb
obs_data['eCO2'] = np.where(obs_data['CO2'].isna(), np.nan, obs_data['CO2'] == 500)

In [10]:
obs_data2 = pd.DataFrame(np.nan, 
    index = pd.MultiIndex.from_frame(obs_data[['Plot','Year']]),
    columns = ['eCO2', 'Tair', 
               'AGBiomass_Spruce', 'AGBiomass_Tamarack', 'AGBiomass_Shrub', # gC m-2
               'AGNPPtoBiomass_Spruce', 'AGNPPtoBiomass_Tamarack', 'AGNPPtoBiomass_Shrub', # NPP to Biomass ratio, yr-1
               'AGNPP_Spruce', 'AGNPP_Tamarack', 'AGNPP_Shrub', 'NPP_moss' # gC m-2 yr-1
               'BGNPP_TreeShrub', # fine root NPP, gC m-2 yr-1
               'BGtoAG_TreeShrub', # fine root NPP to AGNPP ratio
               'AGNPP', # tree + shrub + moss
               'HR', 'NEE']
)

In [11]:
obs_data2.loc[:, 'eCO2'] = obs_data['eCO2'].values
obs_data2.loc[:, 'Tair'] = obs_data.loc[:, 'Mean Annual Temp. at 2 m'].values
obs_data2.loc[:, 'BGNPP_TreeShrub'] = obs_data['BNPP Tree & Shrub'].values
obs_data2.loc[:, 'HR'] = obs_data.loc[:, 'RHCO2'].values
obs_data2.loc[:, 'NEE'] = obs_data.loc[:, 'NCE'].values
obs_data2.loc[:, 'NPP_moss'] = obs_data.loc[:, 'NPP Sphag.'].values

In [12]:
# Use linear regression to remove pre-treatment effect on the
# aboveground biomass of trees
# 
# Post treatment ~ I(eCO2) + beta1 * Tair + beta2 * Pre treatment + beta3 * Year
#    + I(eCO2) x Tair + I(eCO2) x Pre treatment + I(eCO2) x Year
# 
# Subtract out (beta2 * Pre treatment + I(eCO2) x Pre treatment) if significant

for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], 
                    ['evergreen conifer','deciduous conifer','shrub']):
    temp_df = {
        'eCO2': obs_data['eCO2'],
        'Tair': obs_data['Mean Annual Temp. at 2 m'].values,
        'Pretrt': obs_data[('Pretrt_ABGbiomass_gCperm2',pft2)].values,
        'Year': obs_data['Year'].values,
        'Postrt': obs_data[('ABGbiomass_gCperm2',pft2)].values,
    }
    temp_df = pd.DataFrame(temp_df)
    temp_df = temp_df.dropna(how = 'any', axis = 0)

    # add only the significant interactions, identified from an 
    # initial pass that includes all three interactions
    if pft == 'Spruce':
        temp_df['eCO2_Pretrt'] = temp_df['eCO2'] * temp_df['Pretrt']
    elif pft == 'Tamarack':
        temp_df['eCO2_Year'] = temp_df['eCO2'] * temp_df['Year']
    elif pft == 'Shrub':
        pass

    X = sm.add_constant(temp_df.drop('Postrt', axis = 1))
    y = temp_df['Postrt']

    model = sm.OLS(y, X)
    results = model.fit()

    # print the regression outcomes
    print(results.summary())

    # save the subtracted data
    # Shrub pre treatment effect is insignificant
    if pft == 'Spruce':
        obs_data2.loc[:, f'AGBiomass_{pft}'] = \
            obs_data[('ABGbiomass_gCperm2', pft2)].values - \
            results.params['Pretrt'] * \
                (obs_data[('Pretrt_ABGbiomass_gCperm2', pft2)].values - \
                 obs_data[('Pretrt_ABGbiomass_gCperm2', pft2)].mean()) - \
            results.params['eCO2_Pretrt'] * obs_data['eCO2'].values * \
                (obs_data[('Pretrt_ABGbiomass_gCperm2', pft2)].values - \
                 obs_data[('Pretrt_ABGbiomass_gCperm2', pft2)].mean())

    elif pft == 'Tamarack':
        obs_data2.loc[:, f'AGBiomass_{pft}'] = \
            obs_data[('ABGbiomass_gCperm2', pft2)].values - \
            results.params['Pretrt'] * \
                (obs_data[('Pretrt_ABGbiomass_gCperm2', pft2)].values - \
                 obs_data[('Pretrt_ABGbiomass_gCperm2', pft2)].mean())

    elif pft == 'Shrub':
        obs_data2.loc[:, f'AGBiomass_{pft}'] = \
            obs_data[('ABGbiomass_gCperm2', pft2)].values

                            OLS Regression Results                            
Dep. Variable:                 Postrt   R-squared:                       0.982
Model:                            OLS   Adj. R-squared:                  0.979
Method:                 Least Squares   F-statistic:                     467.3
Date:                Wed, 06 Nov 2024   Prob (F-statistic):           6.07e-37
Time:                        14:49:39   Log-Likelihood:                -288.62
No. Observations:                  50   AIC:                             589.2
Df Residuals:                      44   BIC:                             600.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const       -1.215e+05   1.38e+04     -8.827      

In [13]:
for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], 
                    ['evergreen conifer','deciduous conifer','shrub']):
    obs_data2.loc[:, f'AGNPPtoBiomass_{pft}'] = \
        obs_data[('ABGnpp_gCperm2peryear', pft2)].values / \
        obs_data[('ABGbiomass_gCperm2', pft2)].values
    obs_data2.loc[:, f'AGNPP_{pft}'] = obs_data[('ABGnpp_gCperm2peryear', pft2)].values

In [14]:
obs_data2.loc[:, 'BGtoAG_TreeShrub'] = \
    obs_data2.loc[:, 'BGNPP_TreeShrub'].values / \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub']).values

In [15]:
obs_data2.loc[:, 'NPP'] = \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub'] + obs_data2.loc[:, 'BGNPP_TreeShrub'] + \
     obs_data2.loc[:, 'NPP_moss']).values

In [16]:
path_out = os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 'extract')
obs_data2.to_csv(os.path.join(path_out, 'extract_obs_productivity.csv'))

obs_data2

eCO2       Tair  AGBiomass_Spruce  AGBiomass_Tamarack  \
Plot Year                                                          
P04  2016   1.0  12.700000        897.774444          210.920777   
     2017   1.0  11.300000        965.395444          228.653777   
     2018   1.0  10.800000       1054.306444          244.593777   
     2019   1.0   9.967064       1116.127444          267.961777   
     2021   1.0  12.100000       1186.455444          255.372777   
...         ...        ...               ...                 ...   
P13  2020   NaN        NaN               NaN          299.153970   
P16  2020   NaN        NaN               NaN          208.669694   
P17  2020   NaN        NaN               NaN          271.996316   
P19  2020   NaN        NaN               NaN          231.204523   
P20  2020   NaN        NaN               NaN          314.811348   

           AGBiomass_Shrub  AGNPPtoBiomass_Spruce  AGNPPtoBiomass_Tamarack  \
Plot Year                                                                    
P04  2016          354.660               0.161327                 0.007305   
     2017          418.203               0.099601                 0.109131   
     2018          273.278               0.150335                 0.145624   
     2019          311.169               0.121375                 0.165687   
     2021          423.661               0.128676                 0.108055   
...                    ...                    ...                      ...   
P13  2020              NaN               0.104109                 0.129768   
P16  2020              NaN               0.070416                 0.139325   
P17  2020              NaN               0.076110                 0.194170   
P19  2020              NaN               0.062666                 0.103896   
P20  2020              NaN               0.099630                 0.152179   

           AGNPPtoBiomass_Shrub  AGNPP_Spruce  AGNPP_Tamarack  AGNPP_Shrub  \
Plot Year                                                                    
P04  2016              0.499774        76.495           1.385      177.250   
     2017              0.390753        53.962          22.625      163.414   
     2018              0.343566        94.815          32.512       93.889   
     2019              0.406605        84.054          40.863      126.523   
     2021              0.354991        98.159          25.289      150.396   
...                         ...           ...             ...          ...   
P13  2020                   NaN        93.643          26.818          NaN   
P16  2020                   NaN        92.437          53.603          NaN   
P17  2020                   NaN        24.639          69.947          NaN   
P19  2020                   NaN       132.858           0.464          NaN   
P20  2020                   NaN       219.103          84.656          NaN   

           NPP_mossBGNPP_TreeShrub  BGtoAG_TreeShrub  AGNPP         HR  \
Plot Year                                                                
P04  2016                      NaN          0.411163    NaN -537.30000   
     2017                      NaN          0.453331    NaN -453.30000   
     2018                      NaN          0.483012    NaN -456.80000   
     2019                      NaN          0.428830    NaN -381.07052   
     2021                      NaN          0.392856    NaN -453.75570   
...                            ...               ...    ...        ...   
P13  2020                      NaN               NaN    NaN        NaN   
P16  2020                      NaN               NaN    NaN        NaN   
P17  2020                      NaN               NaN    NaN        NaN   
P19  2020                      NaN               NaN    NaN        NaN   
P20  2020                      NaN               NaN    NaN        NaN   

                  NEE  BGNPP_TreeShrub  NPP_moss        NPP  
Plot Year                                                    
P04  2016